In [ ]:
global_step = 0
for epoch in range(cfg.epochs):
    model.train()
    epoch_loss = 0
    train_losses_per_timesteps = [0]*10
    train_losses_per_timesteps_count = [0]*10
    tqdm_bar = tqdm(total=len(train_dataloader), desc="Latent DiT DDPM Training")

    start_at = time.time()
    for idx, data in enumerate(train_dataloader):
        x_0 = data['image'].to(device)
        label_class = data['label'].to(device)

        # # 2) VAE encode
        with torch.no_grad():
            z_0 = vae.encode(x_0)
            z_0 = z_0['latent_dist'].sample() * cfg.latent_scale

        # Flatten latent
        b, c, h, w = z_0.shape
        seq_len = h * w # Define seq_len for train loop too
        z_0 = z_0.permute(0, 2, 3, 1).reshape(b, h*w, c)
        z_T = torch.randn_like(z_0, device=z_0.device, dtype=z_0.dtype)

        # timestep은 0~1000이 아니라 0~1 사이 실수 값 uniform
        if cfg.sampling_method == 'uniform':
            t = torch.rand((b, ), dtype=z_0.dtype, device=device)
        elif cfg.sampling_method == "lognorm":
            if random.random()<0.2:
                t = torch.rand((b, ), dtype=z_0.dtype, device=device)
            else:
                tnorm = np.random.normal(loc=0, scale=1.0, size=b)
                t = 1 / (1 + np.exp(-tnorm))
                t = torch.tensor(t, dtype=z_0.dtype, device=device)

        t = rearrange(t, "b -> b () ()")
        z_t = (1 - (1 - cfg.sigma_min)*t) * z_T + t * z_0
        target_vf = z_0 - (1 - cfg.sigma_min) * z_T

        block_size = 4
        # seq_len is already defined earlier in the loop
        num_blocks = seq_len // block_size

        mask_ratio = random.uniform(0.7, 1.0)
        # block 단위의 마스크 생성: shape (b, num_blocks)
        block_mask = (torch.rand((b, num_blocks), device=z_0.device) > mask_ratio).float()  # 1: keep, 0: mask
        # block mask를 (b, seq_len)로 확장
        mask = block_mask.repeat_interleave(block_size, dim=1)

        # 나머지 짜투리 부분 처리 (예: seq_len이 block_size의 배수가 아닌 경우)
        if mask.shape[1] < seq_len:
            pad = torch.ones((b, seq_len - mask.shape[1]), device=mask.device)
            mask = torch.cat([mask, pad], dim=1)

        context = z_0 * mask.unsqueeze(-1)  # (b, h*w, c)
        img_mask = torch.ones((b, seq_len)).to(device).bool()

        # 4) 모델 예측
        predicted_vf = model(
            w=z_t,
            context=context,
            mask=img_mask,
            times=t.squeeze(),
            cls=label_class
        )

        # label이 다른 데이터 하나를 고른 다음, mse를 계산한뒤 람다만큼 곱해서 뺀다.
        lambda_ = 0.05
        contrastive_error = 0
        for bi in range(predicted_vf.shape[0]):
            pvf = predicted_vf[bi]
            glabel = label_class[bi]
            for li in random.shuffle([li for li in range(label_class.shape[0])]):
                if label_class[li] != glabel:
                    lli = li
                    break
            other_label_vf = target_vf[lli]
            contrastive_error += torch.nn.functional.mse_loss(pvf, other_label_vf, reduction='none')
        # Loss 계산
        # loss = torch.nn.functional.mse_loss(target_vf, predicted_vf, reduction='none').mean(dim=(1, 2))
        # Loss 계산: 마스킹된 위치 (mask == 0)에서만
        loss_mask = (mask == 0).unsqueeze(-1)  # (b, h*w, 1)
        loss = torch.nn.functional.mse_loss(target_vf, predicted_vf, reduction='none')  # (b, h*w, c)
        masked_loss = loss * loss_mask
        loss = masked_loss.sum(dim=(1, 2)) / (loss_mask.sum(dim=(1, 2)) + 1e-8)
        
        masked_contrastive_error = contrastive_error * loss_mask
        masked_contrastive_loss = masked_contrastive_error.sum(dim=(1, 2)) / (loss_mask.sum(dim=(1, 2)) + 1e-8)

        loss = loss - lambda_ * masked_contrastive_loss

        if idx>5:
            for idxin, ptl in enumerate(loss):
                train_losses_per_timesteps[min(math.floor(t[idxin]*10), 9)] += ptl.cpu().detach().item()
                train_losses_per_timesteps_count[min(math.floor(t[idxin]*10), 9)] += 1

        loss = loss.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        tqdm_bar.update()
        tqdm_bar.set_postfix(loss=loss.item())
        epoch_loss += loss.cpu().detach().item()

        tb_writer.add_scalar("Train/StepLoss", loss.item(), global_step)
        global_step += 1

        if idx%100==99:
            print("loss - ", loss)

    tb_writer.add_scalar("Train/EpochLoss", epoch_loss/len(train_dataloader), epoch)

    trainer['train_times'].append(time.time() - start_at)

    trainer['train_losses'].append(epoch_loss/len(train_dataloader))
    trainer['train_losses_per_timesteps'].append(train_losses_per_timesteps)
    train_text = f'Epoch {epoch} Train loss - {epoch_loss / len(train_dataloader)}\n'
    for i in range(10):
        count = train_losses_per_timesteps_count[i]
        avg = train_losses_per_timesteps[i] / count if count > 0 else 0
        train_text += f"timesteps {i/10:.1f} ~ {i/10+0.1:.1f} : {avg:.4f}\n"

    write(train_text)

    plt.plot(trainer['train_losses'])
    plt.savefig(f'{cfg.output_dir}/train_loss.png')
    plt.close()

    torch.cuda.empty_cache()
    valid_step(epoch)

    print("Epoch end")

tb_writer.close()